# v4 timeout patch

This version keeps the successful TRITON_ATTN/vLLM config, but fixes `/predict` Type1 fallback `ReadTimeout` by reducing `GEN_MAX_TOKENS` to 128 and increasing internal vLLM HTTP timeout to 180 seconds.


# EXACT vLLM Kaggle Submission — TRITON_ATTN patched

This version avoids the observed Tesla T4 FlashInfer BatchPrefill crash by forcing `--attention-backend TRITON_ATTN` when the installed vLLM build supports it.


In [1]:
# CELL 0 — Config (vLLM-compliant deployment, Kaggle 2x T4)
RUN_SERVE=True; RUN_API=True; RUN_TUNNEL=True
BASE_MODEL="/kaggle/input/models/qwen-lm/qwen-3/transformers/8b/1"
TYPE1_LORA="/kaggle/input/datasets/nguyenminhtric/exact-test/DATASET_DATA_TYPE_1"
TYPE1_ARTIFACT="/kaggle/input/datasets/nguyenminhtric/exact-test/DATASET_DATA_TYPE_1/full_model_eval_v2_flatten_preds.json"
TYPE1_PATCH="/kaggle/input/datasets/nguyenminhtric/exact-test/type1_fixed_patch/type1_fixed"
EXACT_REPO="/kaggle/input/datasets/nguyenminhtric/exact-test/EXACT/EXACT"
VLLM_HOST="127.0.0.1"; VLLM_PORT=8001; API_PORT=9000
BASE_SERVED_NAME="qwen3-8b-base"; LORA_SERVED_NAME="qwen3-8b-exact-type1"
MAX_MODEL_LEN=4096; GEN_MAX_TOKENS=128
VLLM_HTTP_TIMEOUT=180  # vLLM on 2xT4 can exceed 55s on first LoRA requests
print("config loaded:",VLLM_HOST,VLLM_PORT,API_PORT,BASE_SERVED_NAME,LORA_SERVED_NAME)

# TRITON_ATTN patched version: this notebook forces --attention-backend TRITON_ATTN when supported,
# to avoid the observed FlashInfer BatchPrefill crash on Tesla T4.


config loaded: 127.0.0.1 8001 9000 qwen3-8b-base qwen3-8b-exact-type1


In [2]:
# CELL 1 — Install vLLM via PROVEN ladder from v20 (NOT pip install vllm latest)
import os,sys,subprocess
os.environ["TRANSFORMERS_NO_TF"]="1"; os.environ["USE_TF"]="0"; os.environ["TF_CPP_MIN_LOG_LEVEL"]="3"
os.environ["PROTOCOL_BUFFERS_PYTHON_IMPLEMENTATION"]="python"; os.environ["VLLM_WORKER_MULTIPROC_METHOD"]="spawn"
def _pip(*a): return subprocess.run([sys.executable,"-m","pip","install","--quiet","--break-system-packages"]+list(a),capture_output=True,text=True)
def _clear():
    for m in list(sys.modules):
        if any(x in m for x in ("vllm","transformers","torch._dynamo","torch._inductor")): del sys.modules[m]
def _imp():
    try:
        from vllm import LLM,SamplingParams; return True,""
    except Exception as e: return False,str(e).split("\n")[0][:180]
ok,msg=_imp()
if ok: print("vLLM present:",msg)
else:
    print("installing vLLM... (",msg,")")
    PAIRS=[("4.48.0","0.22.1"),("4.44.2","0.22.1"),("4.57.6","0.22.1"),("4.48.0",""),("4.46.3","0.9.1"),("4.46.3","0.8.5.post1"),("4.44.2","0.8.5.post1"),("4.44.2","0.7.3"),("4.46.3","0.6.6.post1")]
    for tv,vv in PAIRS:
        pkg=f"vllm=={vv}" if vv else "vllm"
        print(f"  try transformers=={tv} + {pkg} ...",end=" ",flush=True)
        _pip("protobuf==5.29.5"); _pip(f"transformers=={tv}"); _pip(pkg); _clear(); ok,msg=_imp()
        print("OK" if ok else f"FAIL ({msg})",flush=True)
        if ok: break
    if not ok: raise RuntimeError("No vLLM+transformers pair worked; restart clean kernel.")
_pip("xformers","fastapi","uvicorn","requests","httptools")
import torch,vllm as _v,transformers as _t
print("torch:",torch.__version__,"| vllm:",_v.__version__,"| tfm:",_t.__version__,"| gpus:",torch.cuda.device_count())
subprocess.run("nvidia-smi",shell=True)

installing vLLM... ( No module named 'vllm' )
  try transformers==4.48.0 + vllm==0.22.1 ... WARNING 06-14 06:06:49 [config.py:70] Support for Transformers v4 is deprecated. The Transformers v4 codepath will become unmaintained in vLLM v0.22.0 and will be removed in vLLM v0.24.0. Please upgrade to Transformers v5: pip install --upgrade transformers
OK
torch: 2.11.0+cu130 | vllm: 0.22.1 | tfm: 4.57.6 | gpus: 2
Sun Jun 14 06:06:58 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.105.08             Driver Version: 580.105.08     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|===

CompletedProcess(args='nvidia-smi', returncode=0)

In [3]:
# CELL 2 — Kaggle/T4 vLLM env and backend choice
# IMPORTANT:
# Previous run loaded vLLM and /v1/models successfully, but /v1/completions crashed in FlashInfer:
# BatchPrefillWithPagedKVCache failed with error invalid argument.
#
# Therefore this version defaults to TRITON_ATTN via CLI --attention-backend when supported.
# If TRITON_ATTN fails to start, try:
#   ATTENTION_BACKEND = "XFORMERS"
# or:
#   ATTENTION_BACKEND = None
# and inspect /kaggle/working/vllm_server.log.

import os, subprocess, sys, textwrap, pathlib, re, time, json, shutil

VLLM_PORT = 8001
API_PORT = 9000

BASE_SERVED_NAME = "qwen3-8b-base"
LORA_SERVED_NAME = "qwen3-8b-exact-type1"

# Main fix for T4 FlashInfer prefill crash.
ATTENTION_BACKEND = "TRITON_ATTN"  # preferred: avoid FLASHINFER BatchPrefill crash on T4

# vLLM 0.22 validates this when chunked prefill is disabled.
# It must be >= MAX_MODEL_LEN, otherwise /v1/models never starts.
MAX_NUM_BATCHED_TOKENS = MAX_MODEL_LEN
# ATTENTION_BACKEND = "XFORMERS"   # fallback if CLI supports it
# ATTENTION_BACKEND = None         # last fallback: let vLLM auto-pick

# Keep envs harmless. In vLLM 0.22.1 these may be ignored/unknown, so CLI arg is the real fix.
os.environ["CUDA_VISIBLE_DEVICES"] = "0,1"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["OMP_NUM_THREADS"] = "1"

# libcuda symlink fix for JIT/linking on Kaggle
subprocess.run("mkdir -p /usr/local/cuda/lib64", shell=True)
subprocess.run("ln -sf /usr/local/nvidia/lib64/libcuda.so.1 /usr/local/cuda/lib64/libcuda.so", shell=True)
os.environ["LD_LIBRARY_PATH"] = "/usr/local/cuda/lib64:/usr/local/nvidia/lib64:" + os.environ.get("LD_LIBRARY_PATH", "")

print("VLLM_PORT:", VLLM_PORT)
print("API_PORT:", API_PORT)
print("ATTENTION_BACKEND:", ATTENTION_BACKEND)
print("MAX_NUM_BATCHED_TOKENS:", MAX_NUM_BATCHED_TOKENS)
print("LD_LIBRARY_PATH:", os.environ["LD_LIBRARY_PATH"][:300])
!ls -l /usr/local/cuda/lib64/libcuda.so
!nvidia-smi

VLLM_HTTP_TIMEOUT = globals().get("VLLM_HTTP_TIMEOUT", 180)
print("VLLM_HTTP_TIMEOUT:", VLLM_HTTP_TIMEOUT)


VLLM_PORT: 8001
API_PORT: 9000
ATTENTION_BACKEND: TRITON_ATTN
MAX_NUM_BATCHED_TOKENS: 4096
LD_LIBRARY_PATH: /usr/local/cuda/lib64:/usr/local/nvidia/lib64:/usr/local/lib/python3.12/dist-packages/cv2/../../lib64:/usr/local/nvidia/lib64:/usr/local/cuda/lib64:/usr/local/cuda/lib64
lrwxrwxrwx 1 root root 36 Jun 14 06:06 /usr/local/cuda/lib64/libcuda.so -> /usr/local/nvidia/lib64/libcuda.so.1
Sun Jun 14 06:06:58 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.105.08             Driver Version: 580.105.08     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================

In [4]:
# CELL 3 — Detect LoRA rank for --max-lora-rank
import json,os
LORA_RANK=16
try:
    cfg=json.load(open(os.path.join(TYPE1_LORA,"adapter_config.json")))
    LORA_RANK=int(cfg.get("r",16)); print("adapter r =",LORA_RANK)
except Exception as e: print("default rank 16:",e)
MAX_LORA_RANK=next(x for x in [8,16,32,64,128] if x>=LORA_RANK)
print("MAX_LORA_RANK =",MAX_LORA_RANK)

adapter r = 16
MAX_LORA_RANK = 16


In [5]:
# CELL 4 — Launch vLLM OpenAI server (TP=2, dynamic LoRA); poll /v1/models
# This cell starts the vLLM OpenAI-compatible server at 127.0.0.1:8001.
# It uses CLI --attention-backend TRITON_ATTN when available to avoid the observed
# FlashInfer BatchPrefillWithPagedKVCache invalid-argument crash on T4.

import os, sys, time, subprocess, pathlib, requests, json, re, signal

def sh(cmd):
    return subprocess.run(cmd, shell=True, text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT).stdout

# kill previous servers cleanly
subprocess.run("pkill -f 'vllm.entrypoints.openai.api_server' || true", shell=True)
subprocess.run("pkill -f 'vllm serve' || true", shell=True)
time.sleep(5)

# detect max LoRA rank from adapter config
adapter_cfg = pathlib.Path(TYPE1_LORA) / "adapter_config.json"
max_lora_rank = 64
if adapter_cfg.exists():
    try:
        cfg = json.loads(adapter_cfg.read_text())
        max_lora_rank = int(cfg.get("r", cfg.get("rank", 64)))
    except Exception as e:
        print("adapter_config read warning:", repr(e))
print("max_lora_rank:", max_lora_rank)

# Read vLLM help to see if --attention-backend exists in this installed version.
print("Checking vLLM api_server help for attention backend option...")
help_text = sh(f"{sys.executable} -m vllm.entrypoints.openai.api_server --help | grep -i 'attention\\|backend' -n | head -80")
print(help_text[-4000:])

supports_attention_backend = "--attention-backend" in help_text

# ROOT FIX: max_num_batched_tokens must be >= max_model_len when chunked prefill is disabled.
# Previous v2 died before /v1/models due to 512 < 4096 validation error.
cmd = [
    sys.executable, "-m", "vllm.entrypoints.openai.api_server",
    "--model", BASE_MODEL,
    "--host", "127.0.0.1",
    "--port", str(VLLM_PORT),
    "--dtype", "half",
    "--tensor-parallel-size", "2",
    "--enable-lora",
    "--lora-modules", f"{LORA_SERVED_NAME}={TYPE1_LORA}",
    "--served-model-name", BASE_SERVED_NAME,
    "--max-model-len", str(MAX_MODEL_LEN),
    "--gpu-memory-utilization", "0.85",
    "--max-num-seqs", "1",
    "--max-num-batched-tokens", str(MAX_NUM_BATCHED_TOKENS),
    "--max-lora-rank", str(max_lora_rank),
    "--enforce-eager",
    "--disable-custom-all-reduce",
    "--generation-config", "vllm",
    "--no-enable-prefix-caching",
    "--no-enable-chunked-prefill",
]

if ATTENTION_BACKEND and supports_attention_backend:
    cmd += ["--attention-backend", ATTENTION_BACKEND]
    print("Using CLI attention backend:", ATTENTION_BACKEND)
elif ATTENTION_BACKEND:
    # Some vLLM builds do not expose the CLI flag. Keep this visible rather than failing silently.
    print("WARNING: --attention-backend not found in api_server help; cannot force", ATTENTION_BACKEND)
    print("If vLLM still picks FLASHINFER and completions crash, install a vLLM build with --attention-backend or patch backend another way.")

env = os.environ.copy()
env["CUDA_VISIBLE_DEVICES"] = "0,1"
env["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
env["TOKENIZERS_PARALLELISM"] = "false"
env["OMP_NUM_THREADS"] = "1"
env["LD_LIBRARY_PATH"] = "/usr/local/cuda/lib64:/usr/local/nvidia/lib64:" + env.get("LD_LIBRARY_PATH", "")

log_path = pathlib.Path("/kaggle/working/vllm_server.log")
log = open(log_path, "w")

print("CMD:")
print(" ".join(cmd))

p_vllm = subprocess.Popen(cmd, stdout=log, stderr=log, env=env)
print("vLLM pid:", p_vllm.pid)

ready = False
for i in range(60):  # up to ~10 minutes; first start compiles/warmups
    time.sleep(10)
    try:
        r = requests.get(f"http://127.0.0.1:{VLLM_PORT}/v1/models", timeout=5)
        print("wait", i, "status", r.status_code, r.text[:500])
        if r.status_code == 200:
            ready = True
            break
    except Exception as e:
        print("wait", i, repr(e))
    # print a small live tail every 6 waits
    if i % 6 == 5 and log_path.exists():
        print("---- vLLM log tail ----")
        print(log_path.read_text(errors="ignore")[-3000:])

print("READY:", ready)
print("=== FINAL vLLM LOG TAIL ===")
print(log_path.read_text(errors="ignore")[-12000:])

if not ready:
    raise RuntimeError("vLLM /v1/models did not become ready; inspect /kaggle/working/vllm_server.log")


max_lora_rank: 16
Checking vLLM api_server help for attention backend option...
87:                     [--override-attention-dtype OVERRIDE_ATTENTION_DTYPE]
99:                     [--attention-backend ATTENTION_BACKEND]
100:                     [--mamba-backend MAMBA_BACKEND]
105:                     [--distributed-executor-backend ['external_launcher', 'mp', 'ray', 'uni']]
116:                     [--dcp-comm-backend {a2a,ag_rs}]
126:                     [--data-parallel-backend DATA_PARALLEL_BACKEND]
132:                     [--all2all-backend {allgather_reducescatter,deepep_high_throughput,deepep_low_latency,flashinfer_all2allv,flashinfer_nvlink_one_sided,flashinfer_nvlink_two_sided,mori,naive,nixl_ep,pplx}]
162:                     [--kv-offloading-backend {lmcache,native}]
163:                     [--offload-backend {auto,prefetch,uva}]
180:                     [--mm-encoder-attn-backend MM_ENCODER_ATTN_BACKEND]
226:                     [--moe-backend {aiter,auto,cutlass,deep_ge

In [6]:
# CELL 5 — Prepare verifier_v35 from patch (verbatim from working notebook)
from pathlib import Path
import shutil,sys,types,importlib.util
WORK=Path("/kaggle/working/exact_runtime"); PATCH=Path(TYPE1_PATCH); assert PATCH.exists(),PATCH
shutil.rmtree(WORK,ignore_errors=True); (WORK/"app/type1_logic").mkdir(parents=True,exist_ok=True)
for p in [WORK/"app/__init__.py",WORK/"app/type1_logic/__init__.py"]: p.write_text("")
for name in ["parser.py","prompt.py","solver.py","verifier_v35.py"]:
    shutil.copy(PATCH/name,WORK/"app/type1_logic"/name)
def get_v35_module():
    if "app.type1_logic.verifier_v35" in sys.modules: return sys.modules["app.type1_logic.verifier_v35"]
    app_dir=WORK/"app"; t1=app_dir/"type1_logic"
    am=types.ModuleType("app"); am.__path__=[str(app_dir)]; sys.modules["app"]=am
    tm=types.ModuleType("app.type1_logic"); tm.__path__=[str(t1)]; sys.modules["app.type1_logic"]=tm
    for name in ["parser","prompt"]:
        spec=importlib.util.spec_from_file_location(f"app.type1_logic.{name}",t1/f"{name}.py")
        mod=importlib.util.module_from_spec(spec); sys.modules[f"app.type1_logic.{name}"]=mod; spec.loader.exec_module(mod)
    spec=importlib.util.spec_from_file_location("app.type1_logic.verifier_v35",t1/"verifier_v35.py")
    v=importlib.util.module_from_spec(spec); sys.modules["app.type1_logic.verifier_v35"]=v; spec.loader.exec_module(v); return v
def call_v35(question,premises,model_answer): return get_v35_module().verify(question,premises,model_answer)
v35=get_v35_module(); print("verifier loaded; has verify:",hasattr(v35,"verify"))

verifier loaded; has verify: True


In [7]:
# CELL 6 — helpers (parser/verifier verbatim) + generate_vllm (replaces PEFT generate)
import re,time,requests
VLLM_COMPLETIONS=f"http://{VLLM_HOST}:{VLLM_PORT}/v1/completions"
def normalize_ans(ans):
    if ans is None: return None
    ans=str(ans).strip(); low=ans.lower()
    if low=="uncertain": return "Unknown"
    if low in ["yes","no","unknown"]: return low.capitalize()
    if ans.upper() in ["A","B","C","D"]: return ans.upper()
    return ans
def extract_final_answer(text):
    m=re.findall(r"Final Answer:\s*([A-D]|Yes|No|Unknown|Uncertain)\b",text,flags=re.I)
    return normalize_ans(m[0]) if m else None
def clean_until_final_answer(gen):
    kept=[]
    for line in gen.splitlines():
        kept.append(line)
        if "Final Answer:" in line: break
    return "\n".join(kept).strip()
def generate_vllm(prompt,max_new_tokens=GEN_MAX_TOKENS):
    payload={"model":LORA_SERVED_NAME,"prompt":prompt,"max_tokens":max_new_tokens,"temperature":0.0,"stop":None}
    t0=time.time(); r=requests.post(VLLM_COMPLETIONS,json=payload,timeout=VLLM_HTTP_TIMEOUT); sec=time.time()-t0
    r.raise_for_status(); gen=r.json()["choices"][0]["text"]
    clean=clean_until_final_answer(gen); return clean,extract_final_answer(clean),sec
def apply_verdict(verdict,current_ans):
    ans=current_ans; premises_used=[]; note=None
    if isinstance(verdict,tuple):
        if len(verdict)>=1 and verdict[0] is not None: ans=verdict[0]
        if len(verdict)>=2: premises_used=verdict[1] or []
        if len(verdict)>=3: note=verdict[2]
    elif isinstance(verdict,dict):
        cand=verdict.get("answer") or verdict.get("pred") or verdict.get("label")
        if cand is not None: ans=cand
        premises_used=verdict.get("premises_used",[]); note=verdict.get("reasoning") or verdict.get("proof")
    elif isinstance(verdict,str): ans=verdict
    return normalize_ans(ans),premises_used,note
print("helpers ready; generation backend = vLLM",VLLM_COMPLETIONS)

helpers ready; generation backend = vLLM http://127.0.0.1:8001/v1/completions


In [8]:
# CELL 7 — Type2 (toy fallback) + MC-aware Type1 prompt + premises_used fallback
import re
def _field(req,name,default=None):
    if isinstance(req,dict): return req.get(name,default)
    return getattr(req,name,default)
def solve_type2(req):
    qid=_field(req,"query_id","unknown"); q=(_field(req,"query","") or "").lower()
    nums=re.findall(r"r\d*\s*=\s*([0-9.]+)\s*ohm",q); v=re.search(r"([0-9.]+)\s*v\b",q)
    if "parallel" in q and "current" in q and len(nums)>=2 and v:
        rs=[float(x) for x in nums[:2]]; V=float(v.group(1)); I=sum(V/r for r in rs)
        ans=str(int(round(I))) if abs(I-round(I))<1e-9 else f"{I:.6g}"
        return {"query_id":qid,"answer":ans,"unit":"A","explanation":"Computed by deterministic formula solver.","premises_used":[],"reasoning":{"type":"cot","steps":["I=U/R per branch","sum branches",f"{ans} A"]}}
    return {"query_id":qid,"answer":"0","unit":"","explanation":"Type2 fallback could not solve confidently.","premises_used":[],"reasoning":None}
def _is_letter_options(o): return bool(o) and all(re.fullmatch(r"[A-Da-d]",str(x).strip() or "") for x in o)
def build_prompt_from_request(req):
    premises=list(_field(req,"premises",[]) or []); query=_field(req,"query","") or ""; options=list(_field(req,"options",[]) or [])
    lines=["You are solving a logic-based educational QA problem. Use only the given premises. Do not use outside knowledge.","","Premises:"]
    for i,p in enumerate(premises,1): lines.append(f"{i}. {p}")
    lines+=["","Question:",query]
    if options and not _is_letter_options(options):
        textual=[str(o) for o in options]
        if not (set(t.lower() for t in textual)<= {"yes","no","uncertain","unknown"}):
            lines+=["","Options:"]+[f"- {o}" for o in textual]
    if _is_letter_options(options): hint="<A, B, C, or D>"
    elif options and not (set(str(o).lower() for o in options)<= {"yes","no","uncertain","unknown"}): hint="<one of the options listed above>"
    else: hint="<Yes, No, or Unknown>"
    lines+=["",f"Reason step by step briefly, cite supporting premises if useful, and End with exactly one line: Final Answer: {hint}"]
    return "\n".join(lines)+"\n"
def extract_final_answer_mc(text,options):
    m=re.findall(r"Final Answer:\s*([A-D]|Yes|No|Unknown|Uncertain)\b",text,flags=re.I)
    if m:
        tok=m[0].strip()
        return tok.upper() if re.fullmatch(r"[A-Da-d]",tok) else normalize_ans(tok)
    return None
_STOP=set("a an the is are was were be been being do does did of to in on at for and or not no if then than that this those these with without as by from into over under all any some every each least most more less which who whom whose what when where why how it its student students premise premises conclusion statement following based given true false answer".split())
def _cw(t): return {w for w in re.findall(r"[a-zA-Z]+",str(t).lower()) if len(w)>3 and w not in _STOP}
def derive_premises_used(clean_output,question,answer,premises):
    idxs=set()
    for m in re.finditer(r"premise[s]?\s*((?:\d+[\s,and&]*)+)",str(clean_output),flags=re.I):
        for n in re.findall(r"\d+",m.group(1)):
            i=int(n)-1
            if 0<=i<len(premises): idxs.add(i)
    if idxs: return sorted(idxs)
    qw=_cw(question); scored=[]
    for i,p in enumerate(premises):
        ov=len(_cw(p)&qw)
        if ov>=2: scored.append((ov,i))
    scored.sort(reverse=True)
    return [i for _,i in scored[:3]]
def predict_type1_live(req):
    qid=_field(req,"query_id","unknown"); options=list(_field(req,"options",[]) or [])
    premises=list(_field(req,"premises",[]) or []); question=_field(req,"query","") or ""
    clean,_ra,latency=generate_vllm(build_prompt_from_request(req))
    raw_ans=extract_final_answer_mc(clean,options) or "Unknown"; raw_ans=normalize_ans(raw_ans)
    try: verdict=call_v35(question,premises,raw_ans)
    except Exception as e: verdict=(None,[],f"verifier_error:{type(e).__name__}")
    final_ans,premises_used,note=apply_verdict(verdict,raw_ans)
    if not premises_used: premises_used=derive_premises_used(clean,question,raw_ans,premises)
    if final_ans=="Unknown" and "Uncertain" in options: final_ans="Uncertain"
    if options and final_ans not in options: final_ans="Uncertain" if "Uncertain" in options else options[0]
    return {"query_id":qid,"answer":final_ans,"unit":"","explanation":"LoRA Final Answer + v35 verifier override when proof is certain.","premises_used":premises_used,"reasoning":{"raw_answer":raw_ans,"verdict":list(verdict) if isinstance(verdict,tuple) else verdict,"note":note,"latency_sec":latency}}
print("Type1 live ready (MC-aware + premises_used fallback)")

Type1 live ready (MC-aware + premises_used fallback)


In [9]:
# CELL 7b — Route Type2 through REAL EXACT physics pipeline (deterministic + vLLM fallback)
import sys,re
from pathlib import Path
if EXACT_REPO not in sys.path: sys.path.insert(0,EXACT_REPO)
PHYS=None; PHYS_BASE_URL=f"http://{VLLM_HOST}:{VLLM_PORT}/v1"
try:
    import subprocess; subprocess.run([sys.executable,"-m","pip","install","-q","--break-system-packages","sympy>=1.12"])
    from model.pipeline import PhysicsPipeline
    from model.config import PipelineConfig
    from model.llm_client import normalize_base_url
    _pc=PipelineConfig(base_url=PHYS_BASE_URL,model=BASE_SERVED_NAME,generator="qwen",prefer_solver=True,skip_model_check=True,enable_thinking=False,temperature=0.0,max_tokens=1024,timeout=50.0,kb_root=Path("knowledge_base/physics"))
    PHYS=PhysicsPipeline(_pc,repo_root=Path(EXACT_REPO)); PHYS_BASE_URL=normalize_base_url(PHYS_BASE_URL)
    print("EXACT physics pipeline READY")
except Exception as e:
    print("physics pipeline NOT loaded -> toy fallback:",type(e).__name__,str(e)[:200])
_NU=re.compile(r"^\s*([-+]?\d[\d.,]*(?:\s*[x×*]\s*10\s*\^?\s*[-+]?\d+)?(?:[eE][-+]?\d+)?)\s*(.*)$")
def _split_nu(s):
    s=str(s).strip(); m=_NU.match(s)
    return (m.group(1).replace(" ",""),m.group(2).strip()) if m else (s,"")
_toy_solve_type2=solve_type2
def solve_type2(req):
    qid=_field(req,"query_id","unknown"); q=_field(req,"query","") or ""
    if PHYS is None: return _toy_solve_type2(req)
    try:
        out=PHYS.run_one({"question":q,"id":qid},0,PHYS_BASE_URL); resp=out.response or {}
        num,unit=_split_nu(resp.get("answer",""))
        if not num or num.lower()=="unknown": return _toy_solve_type2(req)
        steps=[str(s) for s in (resp.get("cot") or [])][:8]
        return {"query_id":qid,"answer":num,"unit":unit,"explanation":resp.get("explanation") or "EXACT physics pipeline.","premises_used":[],"reasoning":{"type":"cot","steps":steps} if steps else None}
    except Exception as e:
        r=_toy_solve_type2(req); r["explanation"]=f"physics pipeline error ({type(e).__name__}); fallback."; return r
print("Type2 routed through EXACT physics pipeline.")

EXACT physics pipeline READY
Type2 routed through EXACT physics pipeline.


In [10]:
# CELL 8 — FastAPI: /predict + /health + proxy /v1/models,/v1/completions,/v1/chat/completions
from fastapi import FastAPI,Request
from fastapi.responses import JSONResponse
from pydantic import BaseModel,Field
from typing import Literal
import requests
VLLM_BASE=f"http://{VLLM_HOST}:{VLLM_PORT}"
class PredictRequest(BaseModel):
    query_id:str; type:Literal["type1","type2"]; query:str
    premises:list[str]=Field(default_factory=list); options:list[str]=Field(default_factory=list)
app=FastAPI(title="EXACT vLLM API")
@app.get("/health")
def health():
    try: return {"status":"ok","vllm":requests.get(f"{VLLM_BASE}/v1/models",timeout=5).status_code}
    except Exception as e: return {"status":"degraded","vllm_error":type(e).__name__}
@app.get("/v1/models")
def pm():
    r=requests.get(f"{VLLM_BASE}/v1/models",timeout=10); return JSONResponse(r.json(),status_code=r.status_code)
@app.post("/v1/completions")
async def pc(req:Request):
    r=requests.post(f"{VLLM_BASE}/v1/completions",json=await req.json(),timeout=VLLM_HTTP_TIMEOUT); return JSONResponse(r.json(),status_code=r.status_code)
@app.post("/v1/chat/completions")
async def pcc(req:Request):
    r=requests.post(f"{VLLM_BASE}/v1/chat/completions",json=await req.json(),timeout=VLLM_HTTP_TIMEOUT); return JSONResponse(r.json(),status_code=r.status_code)
@app.post("/predict")
def predict(req:PredictRequest):
    try:
        return [solve_type2(req)] if req.type=="type2" else [predict_type1_live(req)]
    except Exception as e:
        fb="Uncertain" if "Uncertain" in getattr(req,"options",[]) else "Unknown"
        return [{"query_id":getattr(req,"query_id","unknown"),"answer":fb,"unit":"","explanation":f"fallback {type(e).__name__}","premises_used":[],"reasoning":None}]
print("FastAPI ready (with /v1 proxy)")

FastAPI ready (with /v1 proxy)


In [11]:
# CELL 9 — Start FastAPI in background
import uvicorn,threading,time,requests
if RUN_API:
    if "server" in globals():
        try: server.should_exit=True; time.sleep(2)
        except Exception: pass
    cfg=uvicorn.Config(app,host="0.0.0.0",port=API_PORT,log_level="info"); server=uvicorn.Server(cfg)
    threading.Thread(target=server.run,daemon=True).start(); time.sleep(5)
    print("health:",requests.get(f"http://127.0.0.1:{API_PORT}/health",timeout=10).text)
    print("v1/models:",requests.get(f"http://127.0.0.1:{API_PORT}/v1/models",timeout=10).text[:500])
else: print("RUN_API=False")

INFO:     Started server process [58]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:9000 (Press CTRL+C to quit)


INFO:     127.0.0.1:38048 - "GET /health HTTP/1.1" 200 OK
health: {"status":"ok","vllm":200}
INFO:     127.0.0.1:38052 - "GET /v1/models HTTP/1.1" 200 OK
v1/models: {"object":"list","data":[{"id":"qwen3-8b-base","object":"model","created":1781417398,"owned_by":"vllm","root":"/kaggle/input/models/qwen-lm/qwen-3/transformers/8b/1","parent":null,"max_model_len":4096,"permission":[{"id":"modelperm-8e6fc5eaf1cde767","object":"model_permission","created":1781417398,"allow_create_engine":false,"allow_sampling":true,"allow_logprobs":true,"allow_search_indices":false,"allow_view":true,"allow_fine_tuning":false,"organization":"*","group":null,"is_blocking":false}]},{


In [12]:
# CELL 10 — Local tests: /predict (T1,T2) + PROOF /v1/completions with LoRA
import requests
LOCAL=f"http://127.0.0.1:{API_PORT}/predict"
for p in [
 {"query_id":"T1_yes","type":"type1","query":"Does at least one student receive a scholarship?","premises":["Every student receives a scholarship."],"options":["Yes","No","Uncertain"]},
 {"query_id":"T2","type":"type2","query":"Two resistors R1 = 4 ohm and R2 = 6 ohm are in parallel across a 12 V battery. Find the total current.","premises":[],"options":[]},
]:
    r=requests.post(LOCAL,json=p,timeout=180); print("\n",p["query_id"],r.status_code,r.text[:900])
# PROOF: LoRA generation via /v1/completions (if this fails, Type1 /predict fails)
_c=requests.post(f"http://127.0.0.1:{API_PORT}/v1/completions",json={"model":LORA_SERVED_NAME,"prompt":"Premises:\n1. Every student receives a scholarship.\n\nQuestion:\nDoes at least one student receive a scholarship?\n\nEnd with exactly one line: Final Answer: <Yes, No, or Unknown>\n","max_tokens":64,"temperature":0.0},timeout=60)
print("\nLOCAL /v1/completions:",_c.status_code, (_c.json()["choices"][0]["text"][:300] if _c.status_code==200 else _c.text[:300]))

print("\nATTENTION_BACKEND diagnostic:", globals().get("ATTENTION_BACKEND"))
print("If /v1/completions returned 500 and log shows FLASHINFER BatchPrefill, restart and try ATTENTION_BACKEND='XFORMERS' or verify --attention-backend support.")


INFO:     127.0.0.1:38064 - "POST /predict HTTP/1.1" 200 OK

 T1_yes 200 [{"query_id":"T1_yes","answer":"Yes","unit":"","explanation":"LoRA Final Answer + v35 verifier override when proof is certain.","premises_used":[0],"reasoning":{"raw_answer":"Yes","verdict":["Yes",[0],"PE: positive witness/universal proof establishes existence"],"note":"PE: positive witness/universal proof establishes existence","latency_sec":56.72952651977539}}]
INFO:     127.0.0.1:43460 - "POST /predict HTTP/1.1" 200 OK

 T2 200 [{"query_id":"T2","answer":"5","unit":"A","explanation":"The total current is the sum of the currents through each resistor in parallel.","premises_used":[],"reasoning":{"type":"cot","steps":["Step 1: Compute each branch current using I = U/R.","Step 2: Add branch currents.","Step 3: The result is 5 A."]}}]
INFO:     127.0.0.1:40108 - "POST /v1/completions HTTP/1.1" 200 OK

LOCAL /v1/completions: 200 Reason step by step:

Step 1: From premise 1, we know that every student receives a scho

In [13]:
# CELL 11 — Cloudflare Tunnel (exposes /predict AND /v1/* publicly)
if RUN_TUNNEL:
    import subprocess,time,pathlib,re
    subprocess.run("wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /kaggle/working/cloudflared",shell=True)
    subprocess.run("chmod +x /kaggle/working/cloudflared",shell=True)
    subprocess.run("pkill -f cloudflared || true",shell=True); time.sleep(2)
    cf="/kaggle/working/cloudflared.log"
    subprocess.Popen(["/kaggle/working/cloudflared","tunnel","--url",f"http://127.0.0.1:{API_PORT}","--no-autoupdate"],stdout=open(cf,"w"),stderr=subprocess.STDOUT)
    time.sleep(12); txt=pathlib.Path(cf).read_text()
    m=re.search(r"https://[-a-zA-Z0-9.]+\.trycloudflare\.com",txt)
    if not m: print(txt[-4000:]); raise RuntimeError("no tunnel URL")
    PUBLIC_BASE_URL=m.group(0)
    print("PUBLIC_BASE_URL:",PUBLIC_BASE_URL)
    print("PREDICTION:",PUBLIC_BASE_URL+"/predict"); print("V1_MODELS:",PUBLIC_BASE_URL+"/v1/models")
else: print("RUN_TUNNEL=False")

PUBLIC_BASE_URL: https://logical-defeat-dude-surveys.trycloudflare.com
PREDICTION: https://logical-defeat-dude-surveys.trycloudflare.com/predict
V1_MODELS: https://logical-defeat-dude-surveys.trycloudflare.com/v1/models


In [14]:
# CELL 12 — Public PROOF tests (v1/models + v1/completions LoRA + predict) + write urls.txt
import requests,json
print("health:",requests.get(PUBLIC_BASE_URL+"/health",timeout=30).text)
print("v1/models:",requests.get(PUBLIC_BASE_URL+"/v1/models",timeout=30).text[:600])
_pc=requests.post(PUBLIC_BASE_URL+"/v1/completions",json={"model":LORA_SERVED_NAME,"prompt":"Question: 1+1?\nFinal Answer:","max_tokens":16,"temperature":0.0},timeout=60)
print("public /v1/completions:",_pc.status_code,_pc.text[:300])
for p in [
 {"query_id":"T1_pub","type":"type1","query":"Does at least one student receive a scholarship?","premises":["Every student receives a scholarship."],"options":["Yes","No","Uncertain"]},
 {"query_id":"T2_pub","type":"type2","query":"Two resistors R1 = 4 ohm and R2 = 6 ohm are in parallel across a 12 V battery. Find the total current.","premises":[],"options":[]},
]:
    r=requests.post(PUBLIC_BASE_URL+"/predict",json=p,timeout=180); print("\n",p["query_id"],r.status_code,r.text[:900])
urls=(f"PREDICTION_URL={PUBLIC_BASE_URL}/predict\nV1_MODELS_URL={PUBLIC_BASE_URL}/v1/models\nV1_COMPLETIONS_URL={PUBLIC_BASE_URL}/v1/completions\nHEALTH_URL={PUBLIC_BASE_URL}/health\n")
open("/kaggle/working/urls.txt","w").write(urls); print("\n=== urls.txt ===\n"+urls)

INFO:     136.117.250.17:0 - "GET /health HTTP/1.1" 200 OK
health: {"status":"ok","vllm":200}
INFO:     136.117.250.17:0 - "GET /v1/models HTTP/1.1" 200 OK
v1/models: {"object":"list","data":[{"id":"qwen3-8b-base","object":"model","created":1781417502,"owned_by":"vllm","root":"/kaggle/input/models/qwen-lm/qwen-3/transformers/8b/1","parent":null,"max_model_len":4096,"permission":[{"id":"modelperm-b6058bea46f57f6d","object":"model_permission","created":1781417502,"allow_create_engine":false,"allow_sampling":true,"allow_logprobs":true,"allow_search_indices":false,"allow_view":true,"allow_fine_tuning":false,"organization":"*","group":null,"is_blocking":false}]},{"id":"qwen3-8b-exact-type1","object":"model","created":1781417502,"owned_by":"vllm","root":"/kaggle/
INFO:     136.117.250.17:0 - "POST /v1/completions HTTP/1.1" 200 OK
public /v1/completions: 200 {"id":"cmpl-a9a168c14770f593","object":"text_completion","created":1781417503,"model":"qwen3-8b-exact-type1","choices":[{"index":0,"text

In [15]:
# CELL 13 — Stop (run only when done)
# import subprocess,time
# if "server" in globals(): server.should_exit=True
# subprocess.run("pkill -f cloudflared || true",shell=True); subprocess.run("pkill -f vllm || true",shell=True); print("stopped")

In [16]:
!find /kaggle/working /kaggle/input -iname "*notation*csv" -o -iname "*mapping*csv"

/kaggle/input/datasets/nguyenminhtric/exact-test/EXACT2026_Notation_Mapping_Template.csv


In [19]:
# Fill notation_mapping.csv by scanning the actual running system, not blind copy.

import os, re, json, csv, math, textwrap
from pathlib import Path
import pandas as pd

# ---- paths ----
CANDIDATE_CSVS = [
    "/kaggle/working/notation_mapping.csv",
    "/kaggle/working/EXACT2026_Notation_Mapping_Template.csv",
    "/kaggle/input/datasets/nguyenminhtric/exact-test/EXACT2026_Notation_Mapping_Template.csv",
    "/kaggle/input/datasets/nguyenminhtric/exact-test/EXACT2026_Notation_Mapping_Template (1).csv",
]

template = None
for p in CANDIDATE_CSVS:
    if Path(p).exists():
        template = Path(p)
        break

if template is None:
    # fallback: search
    found = list(Path("/kaggle").rglob("*Notation*Mapping*csv")) + list(Path("/kaggle").rglob("*notation*mapping*csv"))
    print("found:", found[:20])
    if not found:
        raise FileNotFoundError("Cannot find notation mapping template CSV.")
    template = found[0]

print("Using template:", template)

EXACT_REPO = globals().get("EXACT_REPO", "/kaggle/input/datasets/nguyenminhtric/exact-test/EXACT/EXACT")
SCAN_ROOTS = [
    Path(EXACT_REPO),
    Path("/kaggle/working"),
]

# limit files to source/config/text likely to contain notation
EXTS = {
    ".py", ".md", ".txt", ".csv", ".json", ".yaml", ".yml",
    ".ipynb", ".jinja", ".jinja2", ".toml"
}

MAX_FILE_MB = 8

def safe_read(path: Path):
    try:
        if path.stat().st_size > MAX_FILE_MB * 1024 * 1024:
            return ""
        return path.read_text(encoding="utf-8", errors="ignore")
    except Exception:
        return ""

texts = []
file_hits = {}

for root in SCAN_ROOTS:
    if not root.exists():
        continue
    for p in root.rglob("*"):
        if not p.is_file():
            continue
        if p.suffix.lower() not in EXTS:
            continue
        # skip huge logs/cache
        low = str(p).lower()
        if any(x in low for x in ["/.cache/", "__pycache__", "vllm_server.log", "cloudflared.log"]):
            continue
        t = safe_read(p)
        if not t:
            continue
        texts.append(t)
        file_hits[str(p)] = t

BIG = "\n".join(texts)
BIG_LOW = BIG.lower()

print("Scanned files:", len(file_hits))
print("Scanned chars:", len(BIG))

# ---- helper ----
def exists_any(patterns, text=BIG):
    for pat in patterns:
        if re.search(pat, text, flags=re.I | re.M):
            return True
    return False

def literal_exists(s):
    if not s:
        return False
    return s in BIG

# Evidence-aware mapping table.
# Each entry: canonical_latex -> candidate notation(s) used by our system.
# We prefer ASCII/programming notations because solver/API code usually uses these.
manual_candidates = {
    r"\times": ["*", "x", "×"],
    r"\cdot": ["*", "·"],
    r"\div": ["/"],
    r"/": ["/"],
    r"\frac{a}{b}": ["a/b", "/"],
    "+": ["+"],
    "-": ["-"],
    r"\pm": ["+/-", "±"],
    r"\mp": ["-/+", "∓"],
    "=": ["="],
    r"\approx": ["approx", "~", "≈"],
    r"\neq": ["!=", "≠"],
    "<": ["<"],
    ">": [">"],
    r"\leq": ["<=", "≤"],
    r"\geq": [">=", "≥"],
    r"\propto": ["proportional", "propto", "∝"],
    r"\infty": ["inf", "infinity", "∞"],
    "a^b": ["**", "^"],
    "a_b": ["_", "subscript"],
    r"\sqrt{}": ["sqrt", "sqrt()"],
    r"\sqrt[n]{}": ["nth_root", "root"],
    r"\times 10^{n}": ["e", "E", "*10^"],
    r"\alpha": ["alpha"],
    r"\beta": ["beta"],
    r"\gamma": ["gamma"],
    r"\delta": ["delta"],
    r"\Delta": ["delta", "Delta", "d"],
    r"\epsilon": ["epsilon", "eps"],
    r"\varepsilon": ["epsilon", "eps"],
    r"\theta": ["theta"],
    r"\lambda": ["lambda", "wavelength"],
    r"\mu": ["mu", "micro", "u"],
    r"\pi": ["pi", "math.pi"],
    r"\rho": ["rho", "density"],
    r"\sigma": ["sigma"],
    r"\tau": ["tau"],
    r"\phi": ["phi"],
    r"\varphi": ["phi"],
    r"\Phi": ["Phi", "flux"],
    r"\omega": ["omega", "angular_frequency"],
    r"\Omega": ["ohm", "Ω"],
    r"\int": ["integral", "integrate"],
    r"\sum": ["sum", "summation"],
    r"\partial": ["partial"],
    r"\nabla": ["nabla", "gradient"],
    r"\vec{}": ["vector", "vec"],
    r"\hat{}": ["hat", "unit_vector"],
    r"\degree": ["degree", "deg", "°"],
    r"\%": ["%", "percent"],
    r"\angle": ["angle"],
    "V": ["V", "volt", "voltage", "U"],
    "A": ["A", "ampere", "current", "I"],
    "ohm": ["ohm", "Ω", "R"],
    "W": ["W", "watt", "power", "P"],
    "J": ["J", "joule", "energy"],
    "C": ["C", "coulomb", "charge", "q"],
    "F": ["F", "farad", "capacitance"],
    "H": ["H", "henry", "inductance"],
    "T": ["T", "tesla"],
    "Wb": ["Wb", "weber"],
    "N": ["N", "newton", "force"],
    "V/m": ["V/m", "N/C", "electric field", "E"],
    "N/C": ["N/C", "V/m", "electric field", "E"],
    "Hz": ["Hz", "hertz", "frequency", "f"],
    "s": ["s", "sec", "second", "time", "t"],
    "m": ["m", "meter", "distance", "length"],
    "eV": ["eV", "electronvolt"],
    "p": ["p", "pico"],
    "n": ["n", "nano"],
    "u": ["u", "micro", "µ"],
    "k": ["k", "kilo"],
    "M": ["M", "mega"],
    "G": ["G", "giga"],
}

# More conservative unit/variable evidence:
# If formula solver contains R1/R2/U/I patterns, use the actual solver notation.
def pick_from_candidates(canonical, meaning):
    candidates = manual_candidates.get(str(canonical), [str(canonical)])
    evidence = []

    # Direct literal evidence first.
    for cand in candidates:
        if cand in ["*", "+", "-", "/", "=", "<", ">", "<=", ">=", "!=", "**", "^", "_", "%"]:
            # Operators are almost certainly used in code; still check.
            if cand in BIG:
                evidence.append(cand)
        else:
            pat = r"(?<![A-Za-z0-9_])" + re.escape(cand) + r"(?![A-Za-z0-9_])"
            if re.search(pat, BIG, flags=re.I):
                evidence.append(cand)

    # Semantic overrides based on actual physics/code patterns.
    m = str(meaning).lower()
    c = str(canonical)

    # Voltage: our examples and solver used U/R and "12 V", but unit is V.
    if c == "V":
        if re.search(r"\bU\s*/\s*R\b|\bvoltage\b|\bvolt", BIG, re.I):
            return "V; U", "evidence: voltage/volt or U/R found"

    if c == "A":
        if re.search(r"\bI\s*=\s*U\s*/\s*R\b|\bcurrent\b|\bamp", BIG, re.I):
            return "A; I", "evidence: current/I or ampere found"

    if c == "ohm":
        if re.search(r"\bohm\b|Ω|\bR1\b|\bR2\b|\bresistance\b", BIG, re.I):
            return "ohm; R", "evidence: ohm/R/resistance found"

    if c == "C":
        if re.search(r"\bcoulomb\b|\bcharge\b|q1|q2|\bq\b", BIG, re.I):
            return "C; q", "evidence: coulomb/charge/q found"

    if c == "N":
        if re.search(r"\bnewton\b|\bforce\b|\bF\s*=", BIG, re.I):
            return "N; F", "evidence: force/F/newton found"

    if c in ["V/m", "N/C"]:
        if re.search(r"\belectric field\b|\bE\s*=", BIG, re.I):
            return c + "; E", "evidence: electric field/E found"

    if c == "W":
        if re.search(r"\bpower\b|\bP\s*=", BIG, re.I):
            return "W; P", "evidence: power/P found"

    if c == "F":
        if re.search(r"\bfarad\b|\bcapacitance\b|\bC\s*=", BIG, re.I):
            return "F; C_cap", "evidence: capacitance/farad found"

    if c == "Hz":
        if re.search(r"\bfrequency\b|\bf\s*=", BIG, re.I):
            return "Hz; f", "evidence: frequency/f found"

    if c == "s":
        if re.search(r"\btime\b|\bt\s*=", BIG, re.I):
            return "s; t", "evidence: time/t found"

    if c == "m":
        if re.search(r"\bdistance\b|\blength\b|\bradius\b|\br\s*=", BIG, re.I):
            return "m; r", "evidence: distance/length/r found"

    if evidence:
        # Keep alternatives, first is what portal uses by default.
        # Make the first one the safest ASCII notation.
        return "; ".join(dict.fromkeys(evidence)), "evidence: literal/system occurrence"

    # If no evidence, do NOT leave blank: use conservative canonical symbol, but mark note.
    return str(canonical), "fallback: not found in scanned system; canonical retained"

df = pd.read_csv(template)
required = ["canonical_latex", "meaning", "your_notation"]
assert list(df.columns) == required, f"Bad columns: {df.columns}"

notes = []
notations = []

for _, row in df.iterrows():
    notation, note = pick_from_candidates(row["canonical_latex"], row["meaning"])
    notations.append(notation)
    notes.append(note)

df["your_notation"] = notations

# Write official 3-column file.
out = Path("/kaggle/working/notation_mapping_filled.csv")
df.to_csv(out, index=False)

# Write audit file with evidence notes.
audit = df.copy()
audit["evidence_note"] = notes
audit_out = Path("/kaggle/working/notation_mapping_filled_audit.csv")
audit.to_csv(audit_out, index=False)

print("WROTE:", out)
print("WROTE AUDIT:", audit_out)
print("empty your_notation:", df["your_notation"].astype(str).str.strip().eq("").sum())
print("shape:", df.shape)
print(df.head(30).to_string(index=False))

print("\nRows with fallback canonical retained:")
fallback_rows = audit[audit["evidence_note"].str.startswith("fallback")]
print(fallback_rows[["canonical_latex", "meaning", "your_notation", "evidence_note"]].to_string(index=False))

Using template: /kaggle/input/datasets/nguyenminhtric/exact-test/EXACT2026_Notation_Mapping_Template.csv
Scanned files: 82
Scanned chars: 5669979
WROTE: /kaggle/working/notation_mapping_filled.csv
WROTE AUDIT: /kaggle/working/notation_mapping_filled_audit.csv
empty your_notation: 0
shape: (74, 3)
canonical_latex                meaning           your_notation
         \times         multiplication                 *; x; ×
          \cdot     dot multiplication                    *; ·
           \div               division                       /
              /         fraction slash                       /
    \frac{a}{b}               fraction                  a/b; /
              +               addition                       +
              -    subtraction / minus                       -
            \pm             plus-minus                  +/-; ±
            \mp             minus-plus                  -/+; ∓
              =                 equals                       =
        \